# Music Chart Semantic Search with the Deezer API and Mistral AI

**Author:** Côme Rossary — GitHub: [@comersy](https://github.com/comersy) — Stanford | ISAE-SUPAERO

Explore current music charts the way you would talk to a friend who knows everything about music. Instead of scrolling through a playlist, just ask:

- *"I want something relaxed with a good BPM"*
- *"Give me a song to dance to"*
- *"I need something calm to focus, not too many lyrics"*

This notebook builds a small end-to-end RAG pipeline:

1. **Fetch** — pull the Deezer *Top World* chart (public API, no authentication required)
2. **Enrich** — use Mistral **structured outputs** to generate a typed musical profile for every track (genre, mood, tempo, energy, listening context)
3. **Index** — embed the profiles and store them in ChromaDB for semantic search
4. **Chat** — ask questions in natural language; the most relevant tracks are retrieved and Mistral answers conversationally

The full project (CLI version) lives at [comersy/deezer-llm-search](https://github.com/comersy/deezer-llm-search).

## Setup

Install the pinned dependencies. This notebook is designed to run on Google Colab.

In [4]:
# requests is preinstalled on Colab (pinned by the platform), so we only install chromadb.
# The opentelemetry warnings pip may print are harmless resolver notices.
%pip install -q chromadb==1.5.9

Set your Mistral API key — create a free key at [console.mistral.ai](https://console.mistral.ai). On Colab, the recommended way is to store it as a **Secret**: click the key icon in the left sidebar, add a secret named `MISTRAL_API_KEY`, and enable notebook access. The cell below reads it automatically, and falls back to a masked prompt otherwise.

In [5]:
import os
from google.colab import userdata

# Reads the MISTRAL_API_KEY secret (key icon in the Colab left sidebar)
os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")

MISTRAL_URL = "https://api.mistral.ai/v1/chat/completions"
MISTRAL_MODEL = "mistral-small-latest"

## Step 1 — Fetch the charts

Deezer exposes rich public endpoints with no authentication: charts, genres, artists, playlists. We pull the *Top World* chart and keep a compact record per track.

We limit the demo to 25 tracks so the enrichment step stays fast and cheap — set `N_TRACKS = 100` for the full chart, or point `CHART_URL` at any public Deezer playlist.

In [11]:
import requests

N_TRACKS = 25  # increase up to 100 for the full chart
CHART_URL = f"https://api.deezer.com/playlist/3155776842/tracks?limit={N_TRACKS}"


def fetch_chart(url: str = CHART_URL) -> list[dict]:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    tracks = [
        {
            "id": t["id"],
            "title": t["title"],
            "artist": t["artist"]["name"],
            "album": t["album"]["title"],
            "duration": t["duration"],
        }
        for t in response.json().get("data", [])
    ]
    print(f"{len(tracks)} tracks loaded")
    return tracks


tracks = fetch_chart()
tracks[0]

25 tracks loaded


{'id': 3895241341,
 'title': 'Boston',
 'artist': 'STELLA LEFTY',
 'album': 'Boston',
 'duration': 170}

## Step 2 — Enrich every track with structured outputs

Rather than free-form text, we ask Mistral for a **typed JSON profile** per track using the `json_schema` response format. Structured outputs make the enrichment reliable and directly usable as search metadata — no parsing surprises.

If the account or model does not support `json_schema`, the code automatically falls back to `json_object` mode with the schema described in the prompt.

In [12]:
import json
import time

TRACK_SCHEMA = {
    "type": "object",
    "properties": {
        "genre": {"type": "string", "description": "Main musical genre"},
        "mood": {"type": "string", "description": "Dominant mood, e.g. euphoric, melancholic, chill"},
        "tempo": {"type": "string", "enum": ["slow", "mid", "fast"]},
        "energy": {"type": "integer", "minimum": 1, "maximum": 10},
        "vocals": {"type": "string", "enum": ["instrumental", "few lyrics", "vocal-heavy"]},
        "listening_contexts": {
            "type": "array",
            "items": {"type": "string"},
            "description": "2-3 typical contexts, e.g. party, workout, focus, late-night drive",
        },
        "summary": {"type": "string", "description": "One factual sentence about the track"},
    },
    "required": ["genre", "mood", "tempo", "energy", "vocals", "listening_contexts", "summary"],
    "additionalProperties": False,
}

PROMPT = (
    'Profile the song "{title}" by {artist} from the album "{album}". '
    "Be concise and factual; if you are unsure about a field, give your best musical estimate."
)


def _call(payload: dict) -> dict:
    response = requests.post(
        MISTRAL_URL,
        headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}"},
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return json.loads(response.json()["choices"][0]["message"]["content"])


def profile_track(track: dict) -> dict:
    prompt = PROMPT.format(**track)
    base = {"model": MISTRAL_MODEL, "messages": [{"role": "user", "content": prompt}]}
    try:  # structured outputs (json_schema)
        return _call({**base, "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "track_profile", "strict": True, "schema": TRACK_SCHEMA},
        }})
    except (requests.HTTPError, json.JSONDecodeError):  # fallback: json_object mode
        base["messages"][0]["content"] += (
            " Reply ONLY with a JSON object with keys: "
            "genre, mood, tempo (slow|mid|fast), energy (1-10), "
            "vocals (instrumental|few lyrics|vocal-heavy), listening_contexts (list), summary."
        )
        return _call({**base, "response_format": {"type": "json_object"}})


def enrich_all(tracks: list[dict]) -> list[dict]:
    enriched = []
    for i, track in enumerate(tracks, 1):
        for attempt in range(3):
            try:
                profile = profile_track(track)
                break
            except requests.HTTPError as e:  # simple backoff on rate limits
                if e.response.status_code == 429 and attempt < 2:
                    time.sleep(2 * (attempt + 1))
                else:
                    raise
        enriched.append({**track, **profile})
        if i % 5 == 0 or i == len(tracks):
            print(f"Enriched {i}/{len(tracks)}")
        time.sleep(0.3)  # be gentle with the API
    return enriched


tracks = enrich_all(tracks)
tracks[0]

Enriched 5/25
Enriched 10/25
Enriched 15/25
Enriched 20/25
Enriched 25/25


{'id': 3895241341,
 'title': 'Boston',
 'artist': 'STELLA LEFTY',
 'album': 'Boston',
 'duration': 170,
 'genre': 'indie rock',
 'mood': 'nostalgic',
 'tempo': 'mid',
 'energy': 6,
 'vocals': 'vocal-heavy',
 'listening_contexts': ['late-night drive',
  'reflective moments',
  'coffee shop background'],
 'summary': 'A melancholic indie rock track with introspective lyrics and a steady mid-tempo beat.'}

## Step 3 — Index the profiles in ChromaDB

Each track becomes one document built from its structured profile, embedded with ChromaDB's default embedding function (`all-MiniLM-L6-v2`). The structured fields are also stored as metadata, which enables exact filtering (e.g. `where={"tempo": "slow"}`) on top of semantic search.

In [13]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
collection = client.get_or_create_collection(
    "tracks", embedding_function=embedding_functions.DefaultEmbeddingFunction()
)


def to_document(t: dict) -> str:
    contexts = ", ".join(t["listening_contexts"])
    return (
        f"{t['title']} by {t['artist']}. {t['genre']}, {t['mood']} mood, "
        f"{t['tempo']} tempo, energy {t['energy']}/10, {t['vocals']}. "
        f"Good for: {contexts}. {t['summary']}"
    )


collection.add(
    ids=[str(t["id"]) for t in tracks],
    documents=[to_document(t) for t in tracks],
    metadatas=[
        {"title": t["title"], "artist": t["artist"], "album": t["album"],
         "genre": t["genre"], "mood": t["mood"], "tempo": t["tempo"],
         "energy": t["energy"], "vocals": t["vocals"]}
        for t in tracks
    ],
)
print(f"Index built with {collection.count()} tracks")


def search(query: str, n_results: int = 5, where: dict | None = None) -> list[dict]:
    results = collection.query(query_texts=[query], n_results=n_results, where=where)
    return [
        {**results["metadatas"][0][i], "document": results["documents"][0][i]}
        for i in range(len(results["ids"][0]))
    ]


search("calm music to focus", n_results=3)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:05<00:00, 14.5MiB/s]


Index built with 25 tracks


[{'energy': 3,
  'album': 'Folded',
  'genre': 'R&B',
  'title': 'Folded',
  'vocals': 'vocal-heavy',
  'artist': 'Kehlani',
  'tempo': 'slow',
  'mood': 'melancholic',
  'document': 'Folded by Kehlani. R&B, melancholic mood, slow tempo, energy 3/10, vocal-heavy. Good for: late-night reflection, introspective moments, chill evening. A slow-tempo R&B track with heavy emotional vocals, exploring themes of heartbreak and vulnerability.'},
 {'genre': 'Pop',
  'artist': 'Bruno Mars',
  'album': 'The Romantic',
  'energy': 9,
  'tempo': 'fast',
  'vocals': 'vocal-heavy',
  'title': 'I Just Might',
  'mood': 'playful',
  'document': "I Just Might by Bruno Mars. Pop, playful mood, fast tempo, energy 9/10, vocal-heavy. Good for: party, dance, celebration. A high-energy pop track featuring Bruno Mars's dynamic vocals and a funk-infused groove."},
 {'title': 'Something To Lose',
  'vocals': 'vocal-heavy',
  'tempo': 'mid',
  'album': 'Is This Heaven?',
  'energy': 4,
  'artist': 'STELLA LEFTY',
 

## Step 4 — Chat with the charts

Each question retrieves the most relevant tracks, which are injected into the system prompt. The conversation history is kept so follow-up questions work naturally.

In [14]:
SYSTEM_PROMPT = """You are a music assistant. The user asks questions about current world music charts.
Based on the tracks provided, give a helpful and conversational answer.
Always mention the artist and title of the songs you recommend.
If the context doesn't contain relevant tracks, say so honestly."""

history: list[dict] = []


def ask(question: str) -> str:
    context = "\n".join(f"- {t['document']}" for t in search(question))
    messages = [
        {"role": "system", "content": f"{SYSTEM_PROMPT}\n\nRelevant tracks from the charts:\n{context}"},
        *history,
        {"role": "user", "content": question},
    ]
    response = requests.post(
        MISTRAL_URL,
        headers={"Authorization": f"Bearer {os.environ['MISTRAL_API_KEY']}"},
        json={"model": MISTRAL_MODEL, "messages": messages},
        timeout=60,
    )
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]
    history.extend([{"role": "user", "content": question},
                    {"role": "assistant", "content": answer}])
    return answer

In [15]:
print(ask("I am looking for something relaxed with a good BPM"))

If you're looking for something relaxed with a good BPM, I'd recommend:

- **Midnight Sun by Zara Larsson** - A dreamy, mid-tempo pop track with a BPM that's perfect for a chill evening or a late-night drive. The soaring vocals and atmospheric production make it a great choice for unwinding.

- **Be By You by Luke Combs** - A warm, mid-tempo country ballad with a steady BPM that's ideal for a relaxing evening or a road trip. Luke Combs' smooth vocals and heartfelt message add to the overall calming vibe.


In [16]:
print(ask("Now I want a song to dance!"))

For dancing, I’d recommend these high-energy tracks from the charts:

- **"I Just Might" by Bruno Mars** – A funky, fast-paced pop banger with explosive energy (9/10) and Bruno’s signature vocal power. Perfect for parties or dance floors!

- **"Dai Dai" by Shakira** – A Latin-pop explosion with a driving rhythm, high energy (9/10), and Shakira’s fiery vocals. Great for workouts, dancing, or just letting loose!

Both tracks are guaranteed to get you moving—pick based on whether you want pop-funk (Bruno) or Latin grooves (Shakira)! 🎶💃


In [17]:
print(ask("Finally, something to work to: slow tempo and not too many lyrics"))

For a focused work session with minimal distractions, try:

- **"Folded" by Kehlani** – A slow-tempo (3/10 energy) R&B track with heavy vocals but a gentle, soothing melody. Perfect for late-night reflection or deep work without overwhelming lyrics.

- **"Man I Need" by Olivia Dean** – A mid-tempo (5/10 energy) soul track with expressive vocals but a smooth, steady groove. Ideal for a chill work environment where you still want a little musical texture.


## Going further

- **Any chart or playlist** — swap `CHART_URL` for Top France (`/chart/23/tracks`), a genre chart, or any public playlist (`/playlist/{id}/tracks`)
- **Hybrid search** — combine semantic queries with metadata filters, e.g. `search("summer vibes", where={"tempo": "fast"})`
- **Personal libraries** — with Deezer OAuth, index a user's own playlists
- **Other sources** — the enrich → index → chat pipeline is source-agnostic: Spotify, Last.fm, or local MP3 tags work the same way

Full CLI version: [comersy/deezer-llm-search](https://github.com/comersy/deezer-llm-search)